# Базовое обучение модели классификации новостей в Google Colab

Этот ноутбук предназначен для запуска в Google Colab и демонстрирует полный базовый пайплайн:

- загрузка датасета из локального файла или по прямой ссылке;
- очистка текстов и меток классов;
- разбиение на `train`, `validation`, `test`;
- построение TF-IDF признаков;
- обучение `LogisticRegression`;
- расчет метрик качества;
- сохранение артефактов и отчетов в структуру проекта.

Во всех ячейках даны подробные комментарии на русском языке, чтобы их можно было использовать в тексте ВКР.

## 1. Установка библиотек

Если ноутбук запускается в чистом окружении Colab, выполните следующую ячейку. Если зависимости уже установлены, шаг можно пропустить.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib requests

## 2. Импорты и вспомогательные функции

В этой ячейке задаются все функции, которые понадобятся для загрузки данных, предобработки, обучения модели и сохранения артефактов.

In [ ]:
from __future__ import annotations

import json
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable
from urllib.parse import urlparse

import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split


TEXT_COLUMN_CANDIDATES = (
    "text",
    "content",
    "article",
    "body",
    "news",
    "title",
    "description",
    "текст",
    "новость",
    "заголовок",
    "описание",
)

LABEL_COLUMN_CANDIDATES = (
    "label",
    "category",
    "class",
    "topic",
    "target",
    "tag",
    "метка",
    "категория",
    "класс",
    "тема",
)


@dataclass(frozen=True)
class NotebookPaths:
    """Хранит основные каталоги проекта, используемые ноутбуком."""

    root: Path
    data_raw_dir: Path
    data_processed_dir: Path
    data_split_dir: Path
    feature_dir: Path
    model_dir: Path
    vectorizer_dir: Path
    report_dir: Path


def build_project_paths(project_root: str | Path) -> NotebookPaths:
    """Создает стандартную структуру директорий проекта внутри Colab или Google Drive."""
    root = Path(project_root).resolve()
    paths = NotebookPaths(
        root=root,
        data_raw_dir=root / "data" / "raw",
        data_processed_dir=root / "data" / "processed",
        data_split_dir=root / "data" / "splits" / "colab_baseline",
        feature_dir=root / "data" / "features" / "colab_baseline",
        model_dir=root / "models" / "classifiers",
        vectorizer_dir=root / "models" / "vectorizers",
        report_dir=root / "reports" / "colab_baseline",
    )
    for directory in asdict(paths).values():
        Path(directory).mkdir(parents=True, exist_ok=True)
    return paths


def sanitize_filename(name: str) -> str:
    """Готовит безопасное имя файла для артефактов обучения."""
    cleaned_name = "".join(
        symbol if symbol.isalnum() or symbol in {"-", "_", "."} else "_"
        for symbol in name.strip()
    )
    return cleaned_name or "artifact"


def download_dataset_from_url(dataset_url: str, target_dir: Path) -> Path:
    """Скачивает CSV-датасет по прямой ссылке и сохраняет его локально."""
    response = requests.get(dataset_url, timeout=120)
    response.raise_for_status()

    parsed_url = urlparse(dataset_url)
    file_name = Path(parsed_url.path).name or "news_dataset.csv"
    if not file_name.lower().endswith(".csv"):
        file_name = f"{file_name}.csv"

    target_path = target_dir / sanitize_filename(file_name)
    target_path.write_bytes(response.content)
    return target_path


def load_dataset(dataset_source: str | Path) -> pd.DataFrame:
    """Загружает CSV-таблицу из локального файла."""
    return pd.read_csv(dataset_source)


def find_required_column(columns: Iterable[str], candidates: tuple[str, ...], logical_name: str) -> str:
    """Ищет нужную колонку по набору алиасов."""
    normalized_mapping = {str(column).strip().casefold(): str(column) for column in columns}
    for candidate in candidates:
        if candidate in normalized_mapping:
            return normalized_mapping[candidate]
    raise ValueError(f"Не найдена обязательная колонка `{logical_name}`.")


def clean_text_value(text: str) -> str:
    """Выполняет базовую очистку текста новости."""
    cleaned_text = text.replace("\u00a0", " ").replace("\t", " ").replace("\n", " ")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text)
    cleaned_text = re.sub(r"\s+([,.;:!?])", r"\1", cleaned_text)
    return cleaned_text.strip().lower()


def clean_label_value(label: str) -> str:
    """Нормализует строковое представление класса."""
    return re.sub(r"\s+", " ", label).strip()


def prepare_dataset(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Нормализует названия колонок, очищает тексты и удаляет дубликаты."""
    text_column = find_required_column(dataframe.columns, TEXT_COLUMN_CANDIDATES, "text")
    label_column = find_required_column(dataframe.columns, LABEL_COLUMN_CANDIDATES, "label")

    prepared_dataframe = dataframe[[text_column, label_column]].copy()
    prepared_dataframe.columns = ["text", "label"]
    prepared_dataframe["text"] = prepared_dataframe["text"].astype("string").fillna("").map(clean_text_value)
    prepared_dataframe["label"] = prepared_dataframe["label"].astype("string").fillna("").map(clean_label_value)

    prepared_dataframe = prepared_dataframe.loc[
        prepared_dataframe["text"].ne("") & prepared_dataframe["label"].ne("")
    ].copy()
    prepared_dataframe = prepared_dataframe.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    if prepared_dataframe.empty:
        raise ValueError("После очистки в датасете не осталось строк для обучения.")
    if prepared_dataframe["label"].nunique() < 2:
        raise ValueError("Для обучения нужны как минимум два различных класса.")
    return prepared_dataframe


def split_dataset(prepared_dataframe: pd.DataFrame, random_state: int = 42):
    """Разбивает датасет на train, validation и test."""
    train_dataframe, temp_dataframe = train_test_split(
        prepared_dataframe,
        test_size=0.30,
        random_state=random_state,
        stratify=prepared_dataframe["label"],
    )
    validation_dataframe, test_dataframe = train_test_split(
        temp_dataframe,
        test_size=0.50,
        random_state=random_state,
        stratify=temp_dataframe["label"],
    )
    return (
        train_dataframe.reset_index(drop=True),
        validation_dataframe.reset_index(drop=True),
        test_dataframe.reset_index(drop=True),
    )


def calculate_metrics(true_labels: list[str], predicted_labels: np.ndarray) -> dict[str, float | int]:
    """Считает основные метрики качества для одной выборки."""
    accuracy = accuracy_score(true_labels, predicted_labels)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": round(float(accuracy), 4),
        "precision_macro": round(float(precision_macro), 4),
        "recall_macro": round(float(recall_macro), 4),
        "f1_macro": round(float(f1_macro), 4),
        "support": len(true_labels),
    }


def run_baseline_training(prepared_dataframe: pd.DataFrame, paths: NotebookPaths, random_state: int = 42) -> dict:
    """Выполняет полный цикл базового обучения и сохраняет все артефакты."""
    train_dataframe, validation_dataframe, test_dataframe = split_dataset(
        prepared_dataframe,
        random_state=random_state,
    )

    train_path = paths.data_split_dir / "train.csv"
    validation_path = paths.data_split_dir / "validation.csv"
    test_path = paths.data_split_dir / "test.csv"
    train_dataframe.to_csv(train_path, index=False)
    validation_dataframe.to_csv(validation_path, index=False)
    test_dataframe.to_csv(test_path, index=False)

    vectorizer = TfidfVectorizer(
        max_features=20000,
        min_df=1,
        max_df=0.95,
        ngram_range=(1, 2),
        lowercase=False,
        sublinear_tf=True,
    )
    train_matrix = vectorizer.fit_transform(train_dataframe["text"].tolist())
    validation_matrix = vectorizer.transform(validation_dataframe["text"].tolist())
    test_matrix = vectorizer.transform(test_dataframe["text"].tolist())

    vectorizer_path = paths.vectorizer_dir / "colab_baseline_vectorizer.joblib"
    joblib.dump(vectorizer, vectorizer_path)

    model = LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        C=1.0,
        random_state=random_state,
    )
    start_time = time.perf_counter()
    model.fit(train_matrix, train_dataframe["label"].tolist())
    training_seconds = round(time.perf_counter() - start_time, 4)

    model_path = paths.model_dir / "colab_baseline_model.joblib"
    joblib.dump(model, model_path)

    train_predictions = model.predict(train_matrix)
    validation_predictions = model.predict(validation_matrix)
    test_predictions = model.predict(test_matrix)

    metrics = {
        "train": calculate_metrics(train_dataframe["label"].tolist(), train_predictions),
        "validation": calculate_metrics(validation_dataframe["label"].tolist(), validation_predictions),
        "test": calculate_metrics(test_dataframe["label"].tolist(), test_predictions),
    }

    training_report = {
        "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "paths": {
            "train_path": str(train_path),
            "validation_path": str(validation_path),
            "test_path": str(test_path),
            "vectorizer_path": str(vectorizer_path),
            "model_path": str(model_path),
        },
        "dataset": {
            "rows": len(prepared_dataframe),
            "class_count": int(prepared_dataframe["label"].nunique()),
            "classes": sorted(prepared_dataframe["label"].astype(str).unique().tolist()),
        },
        "vectorization": {
            "vocabulary_size": len(vectorizer.vocabulary_),
            "train_shape": list(train_matrix.shape),
            "validation_shape": list(validation_matrix.shape),
            "test_shape": list(test_matrix.shape),
        },
        "training": {
            "solver": model.solver,
            "training_seconds": training_seconds,
            "iterations": [int(value) for value in np.atleast_1d(model.n_iter_)],
        },
        "metrics": metrics,
    }

    report_path = paths.report_dir / "colab_baseline_training_report.json"
    report_path.write_text(
        json.dumps(training_report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return training_report


## 3. Конфигурация запуска

Ниже задаются пути проекта и источник датасета. Для Google Colab удобны два сценария:

1. Указать `DATASET_URL`, если CSV доступен по прямой ссылке.
2. Загрузить CSV вручную через панель файлов Colab и указать путь в `LOCAL_DATASET_PATH`.

Если нужен Google Drive, можно предварительно смонтировать диск и задать `PROJECT_ROOT` внутри него.

In [ ]:
# При необходимости можно раскомментировать строки ниже, чтобы работать через Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_ROOT = "/content/isnews_colab_project"
DATASET_URL = ""
LOCAL_DATASET_PATH = ""

paths = build_project_paths(PROJECT_ROOT)
print("Каталог проекта:", paths.root)
print("Каталог отчетов:", paths.report_dir)

## 4. Загрузка и подготовка датасета

Эта ячейка выбирает источник данных, сохраняет исходный CSV в `data/raw`, затем выполняет очистку и записывает подготовленный датасет в `data/processed`.

In [ ]:
if DATASET_URL:
    raw_dataset_path = download_dataset_from_url(DATASET_URL, paths.data_raw_dir)
elif LOCAL_DATASET_PATH:
    raw_dataset_path = Path(LOCAL_DATASET_PATH).resolve()
else:
    raise ValueError("Укажите DATASET_URL или LOCAL_DATASET_PATH.")

raw_dataframe = load_dataset(raw_dataset_path)
prepared_dataframe = prepare_dataset(raw_dataframe)

processed_dataset_path = paths.data_processed_dir / "colab_prepared_dataset.csv"
prepared_dataframe.to_csv(processed_dataset_path, index=False, encoding="utf-8")

print("Исходный CSV:", raw_dataset_path)
print("Подготовленный CSV:", processed_dataset_path)
print("Размер датасета после очистки:", prepared_dataframe.shape)
prepared_dataframe.head()

## 5. Обучение базовой модели

После запуска следующей ячейки будут сохранены:

- `train.csv`, `validation.csv`, `test.csv`;
- TF-IDF-векторизатор;
- базовая модель `LogisticRegression`;
- JSON-отчет со всеми основными параметрами и метриками.

In [ ]:
baseline_report = run_baseline_training(prepared_dataframe, paths)
print(json.dumps(baseline_report["metrics"], ensure_ascii=False, indent=2))

## 6. Проверка сохраненных артефактов

Завершающая ячейка выводит ключевые пути и позволяет убедиться, что ноутбук сохранил все необходимые файлы для последующего использования в приложении и в ВКР.

In [ ]:
artifact_paths = {
    "processed_dataset": str(paths.data_processed_dir / "colab_prepared_dataset.csv"),
    "train_split": str(paths.data_split_dir / "train.csv"),
    "validation_split": str(paths.data_split_dir / "validation.csv"),
    "test_split": str(paths.data_split_dir / "test.csv"),
    "vectorizer": str(paths.vectorizer_dir / "colab_baseline_vectorizer.joblib"),
    "model": str(paths.model_dir / "colab_baseline_model.joblib"),
    "training_report": str(paths.report_dir / "colab_baseline_training_report.json"),
}

for artifact_name, artifact_path in artifact_paths.items():
    print(f"{artifact_name}: {artifact_path}")